# VRAM probe

Runs a few real training iterations and a few predictions for every model x input size x candidate batch size, and records peak GPU memory and speed. The recommendation table at the end gives, per run, the batch size and gradient accumulation to put into `configs/runs/<run_id>.yaml`, and an estimate of the training time against the 2-hour cap.

- Logic: `iiss.models.probe`. Results: `results/vram_probe/` (keep the outputs of this notebook: they are evidence).
- Run by the user on the local GPU (about 20-30 min for the full matrix). Close other GPU programs first.
- `SMOKE = True`: S=320 only, smallest batch, 5 iterations, 16 images; writes to `runs/smoke/`.

## Parameters

In [ ]:
MODELS = ['yolo', 'maskrcnn', 'm2f']
SIZES = [320, 640, 960]
BATCHES = {'yolo': [4, 8, 16], 'maskrcnn': [1, 2, 4], 'm2f': [1, 2, 4]}
ITERS = 20            # training iterations per probe
WARMUP = 3            # first iterations not timed
INFER_IMAGES = 10     # valid images for the inference timing
BUDGET_MB = 5600      # max training peak (reserved) that counts as fitting on 6 GB
OUT_DIR = 'results/vram_probe'
FRESH = True          # delete the previous probe.csv first
SMOKE = False

In [ ]:
import pandas as pd
import torch
from IPython.display import display

from iiss.models.common import apply_smoke
from iiss.models.probe import probe_matrix, recommend
from iiss.utils.config import load_run_config
from iiss.utils.log import setup_logging
from iiss.utils.paths import repo_root, resolve_path

setup_logging()
ROOT = repo_root()
if SMOKE:
    SIZES, ITERS, WARMUP, INFER_IMAGES = [320], 5, 1, 3
    BATCHES = {m: [min(b)] for m, b in BATCHES.items()}
    OUT_DIR = 'runs/smoke/vram_probe'
out_dir = resolve_path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
if FRESH:
    (out_dir / 'probe.csv').unlink(missing_ok=True)
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Run configs to probe

In [ ]:
configs = []
for model in MODELS:
    for size in SIZES:
        cfg = load_run_config(ROOT / 'configs' / 'runs' / f'{model}_s{size}_seed42.yaml')
        configs.append(apply_smoke(cfg) if SMOKE else cfg)
[c['run_id'] for c in configs]

## Probe

One row per model x S x batch. Larger batches are skipped once a smaller one runs out of memory.

In [ ]:
results = probe_matrix(configs, BATCHES, out_dir / 'probe.csv',
                       iters=ITERS, warmup=WARMUP, infer_images=INFER_IMAGES)
display(results)

## Recommendation

`batch` keeps the configured effective batch (with `accumulate`); `max_fit_batch` is the largest batch that fits at all. `est_train_h` is a planning estimate for the full epoch budget including validation; `cap_risk` = above 2 h.

In [ ]:
rec = recommend(results, {c['run_id']: c for c in configs}, budget_mb=BUDGET_MB)
rec.to_csv(out_dir / 'recommendation.csv', index=False)
display(rec)

## Next step

Send `results/vram_probe/recommendation.csv` (or this table) back to Claude Code. Batch overrides then go into `configs/runs/<run_id>.yaml`, and the epoch budgets are confirmed or adjusted per model (identical across S).